In [ ]:
import warnings

from pygments.lexer import include

warnings.filterwarnings('ignore')

### Установим красивые дефолтные настройки
### Может быть лень постоянно прописывать
### У графиков параметры цвета, размера, шрифта
### Можно положить их в словарь дефолтных настроек

import matplotlib as mlp

mlp.rcParams['lines.linewidth'] = 5
mlp.rcParams['xtick.major.size'] = 20
mlp.rcParams['xtick.major.width'] = 5
mlp.rcParams['xtick.labelsize'] = 20
mlp.rcParams['xtick.color'] = '#FF5533'

mlp.rcParams['ytick.major.size'] = 20
mlp.rcParams['ytick.major.width'] = 5
mlp.rcParams['ytick.labelsize'] = 20
mlp.rcParams['ytick.color'] = '#FF5533'

mlp.rcParams['axes.labelsize'] = 20
mlp.rcParams['axes.titlesize'] = 20
mlp.rcParams['axes.titlecolor'] = '#00B050'
mlp.rcParams['axes.labelcolor'] = '#00B050'

## Практика: Sberbank Russian Housing Market

<div>
<img src="Original.png" width="1000"/>
</div>

In [ ]:
import pandas as pd
import numpy as np

pd.options.display.max_columns = 500

df = pd.read_csv("train.csv")

In [ ]:
df.head()

In [ ]:
df = df.drop(['ID_metro',
 'ID_railroad_station_walk',
 'ID_railroad_station_avto',
 'ID_big_road1',
 'ID_big_road2',
 'ID_railroad_terminal',
 'ID_bus_terminal'], axis=1)

In [ ]:
df = df.assign(log_price_doc=np.log1p(df['price_doc']))
df = df.drop('price_doc', axis=1)

In [ ]:
df

In [ ]:
numeric_columns = df.loc[:, df.dtypes != np.object_].columns
df.describe()

In [ ]:
numeric_columns

In [ ]:
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].mean())


In [ ]:
df[numeric_columns].corr()

In [ ]:
### Секретные функции для фильтрации признаков

def get_redundant_pairs(df):
    pairs_to_drop = set()
    cols = df.columns
    for i in range(0, df.shape[1]):
        for j in range(0, i + 1):
            pairs_to_drop.add((cols[i], cols[j]))
    return pairs_to_drop


def get_top_abs_correlations(df, n=5):
    au_corr = df.corr().abs().unstack()
    labels_to_drop = get_redundant_pairs(df)
    au_corr = au_corr.drop(labels=labels_to_drop).sort_values(ascending=False)
    return au_corr[0:n]


print("Top Absolute Correlations")
print(get_top_abs_correlations(df[numeric_columns], 50))

In [ ]:
print(get_top_abs_correlations(df[numeric_columns], 50))

In [ ]:
### Сворованный со stackoverflow код
### Удалим колонки, где корреляция оказывается > 0.9

def correlation(dataset, threshold=0.9):
    # оставляем только числа
    dataset = dataset.select_dtypes(include=[np.number])

    col_corr = set()
    corr_matrix = dataset.corr()

    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if (corr_matrix.iloc[i, j] >= threshold) and (corr_matrix.columns[j] not in col_corr):
                colname = corr_matrix.columns[i]
                col_corr.add(colname)

    return dataset.drop(columns=list(col_corr))

df = correlation(df, 0.9)

In [ ]:
numeric_columns = df.loc[:,df.dtypes!=np.object_].columns

df.shape

In [ ]:
from sklearn.feature_selection import VarianceThreshold

cutter = VarianceThreshold(threshold=0.1)

In [ ]:

cutter.fit(df[numeric_columns])
constant_cols = [x for x in numeric_columns if x not in cutter.get_feature_names_out()]

df[constant_cols]

In [ ]:
### Посмотрим на категориальные колонки

categorical_columns = df.loc[:,df.dtypes==np.object_].columns

### Изучим их

categorical_columns

In [ ]:
for col in categorical_columns:
    if col != 'timestamp':
        if df[col].nunique() < 5:
            one_hot = pd.get_dummies(df[col], prefix=col, drop_first=True)
            df = pd.concat((df.drop(col, axis=1), one_hot), axis=1)

        else:
            mean_target = df.groupby(col)['log_price_doc'].mean()
            df[col] = df[col].map(mean_target)

In [ ]:
### Поработаем с датой

df['timestamp'] = pd.to_datetime(df['timestamp'])

df['month'] = df.timestamp.dt.month
df['year'] = df.timestamp.dt.year

In [ ]:
df.head()

In [ ]:
### Отсортируем по timestamp
### Потом объясним зачем

df = df.sort_values('timestamp')

df.head()

In [ ]:
### Порисуем графики распределений для некоторых фичей
### Конечно, лучше проводить анализ сразу всех
### Но у нас их слишком много, чтобы поместилось в один урок
### Посмотрим хотя бы на некоторые элементы


### Например, распределения таргета по годам

import matplotlib.pyplot as plt
import seaborn as sns

fig = plt.figure()
fig.set_size_inches(16, 10)

sns.boxplot(y='log_price_doc', x=df['year'].astype('category'), data=df)
plt.show()

In [ ]:
### Закодируем колонку с годом через One-Hot

one_hot = pd.get_dummies(df['year'], prefix='year', drop_first=True)
df = pd.concat((df.drop('year', axis=1), one_hot), axis=1)

In [ ]:
### Распределения таргета по месяцам

fig = plt.figure()
fig.set_size_inches(16, 10)

sns.boxplot(y='log_price_doc', x=df['month'].astype('category'), data=df)
plt.show()

In [ ]:
### Закодируем колонку с месяцем через One-Hot

one_hot = pd.get_dummies(df['month'], prefix='month', drop_first=True)
df = pd.concat((df.drop('month', axis=1), one_hot), axis=1)

In [ ]:
### Распределения таргета по этажу

fig = plt.figure()
fig.set_size_inches(16, 10)

sns.boxplot(y='log_price_doc', x=df['floor'].astype('category'), data=df)
plt.show()

In [ ]:
plt.show()
### Распределения таргета по количеству магазинов в радиусе 3км

fig = plt.figure()
fig.set_size_inches(16, 10)

sns.boxplot(y='log_price_doc', x=df['market_count_3000'].astype('category'), data=df)
plt.show()

In [ ]:
### Распределения таргета по типу недвижимости

fig = plt.figure()
fig.set_size_inches(16, 10)

sns.boxplot(y='log_price_doc', x=df['product_type_OwnerOccupier'].astype('category'), data=df)
plt.show()

In [ ]:
### Уберем timestamp

df = df.drop('timestamp', axis=1)

In [ ]:
### Отделим таргеты от объектов

X = df.drop('log_price_doc', axis=1)
Y = df['log_price_doc']

## Построим пару базовых моделей в качестве бэйзлайна

## Как такую модель валидировать? Можно ли сделать как и ранее?


Теперь наши данные обладают временной структурой. Поэтому, чтобы получить хорошую обобщающую способность, мы хотим построить не просто модель, хорошо работающую на новых данных, а модель, которая угадывает распределение данных в будущем хотя бы на коротком горизонте. Поэтому, когда мы валидируем дизайн модели, нам важно делить на каждом шаге трейн и тест таким образом, чтобы по временной шкале они не пересекались, и точки из второго множества появлялись позже точек из первого.


<div>
<img src="Рисунок7.png" width="500"/>
</div>



Установим "Тренировочную базу" - некоторое множество $\{x_t: x_t\in X, t <= T_0 \}$. Далее на каждом шаге будем отсутпать от него на некоторый фиксированный (для простоты) интервал $T_1$, называя все объекты, которые в него попали, валидацией на текущем шаге. После обучения модели и замера качества, будем добавлять $T_1$ к тренировочной базе. Новую модель будем обучать на более широком трейне, а тест - на более далеких во времени данных.

P.S. тип валидации стоит выбирать, исходя из задачи. Если нам важно хорошо предсказывать что-то для объектов из будущего, то TimeSplit - хорошая идея.

In [ ]:
### Разделим выборку на валидацию и тест

from sklearn.model_selection import TimeSeriesSplit

splitter = TimeSeriesSplit(n_splits=4)

In [ ]:
### Конструкция для замера качества на Кросс-Валидации

from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression


test_losses = []
train_losses = []

for train_index, test_index in splitter.split(X):

    x_train, x_test = X.values[train_index], X.values[test_index]
    y_train, y_test = Y.values[train_index], Y.values[test_index]

    model = LinearRegression()
    model.fit(x_train, y_train)

    preds_test = model.predict(x_test)
    preds_train = model.predict(x_train)

    error_test = np.mean((preds_test - y_test)**2)
    error_train = np.mean((preds_train - y_train)**2)

    test_losses.append(error_test)
    train_losses.append(error_train)

print(f"Среднее MSLE на тренировочных фолдах: {np.mean(train_losses).round(3)}")
print(f"Среднее MSLE на тестовых фолдах: {np.mean(test_losses).round(3)}")

In [ ]:
### Функция cross-validate

from sklearn.model_selection import cross_validate

model = LinearRegression()

cv_result = cross_validate(model, X, Y,
                           scoring='neg_mean_squared_error',
                           cv=splitter, return_train_score=True)

cv_result

In [ ]:
### Убедимся, что результаты совпадают!

print(f"Среднее MSLE на тренировочных фолдах: {-np.mean(cv_result['train_score']).round(3)}")
print(f"Среднее MSLE на тестовых фолдах: {-np.mean(cv_result['test_score']).round(3)}")

In [ ]:
### Как справится теперь модель регуляризации?

from sklearn.linear_model import Lasso, Ridge

model_lasso = Lasso(max_iter=100000)

cv_result_lasso = cross_validate(model_lasso, X, Y,
                                 scoring='neg_mean_squared_error',
                                 cv=splitter, return_train_score=True)

print(f"Среднее MSLE на тренировочных фолдах: {-np.mean(cv_result_lasso['train_score']).round(3)}")
print(f"Среднее MSLE на тестовых фолдах: {-np.mean(cv_result_lasso['test_score']).round(3)}")

In [ ]:
### Как добавить этап нормировки данных в регуляризации?

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([('scaler', StandardScaler()), ('Lasso', Lasso(max_iter=100000))])
pipe.fit(X, Y)

print(pipe.predict(X.head(1)))

cv_result_pipe = cross_validate(pipe, X, Y,
                                scoring='neg_mean_squared_error',
                                cv=splitter, return_train_score=True)

In [ ]:
print(f"Среднее MSLE на тренировочных фолдах: {-np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Среднее MSLE на тестовых фолдах: {-np.mean(cv_result_pipe['test_score']).round(3)}")

In [ ]:
### Какие параметры в нашим лего-конструкторе?

pipe.get_params()

In [ ]:
alphas = np.linspace(start=0.01, stop=1, num=30)
alphas

In [ ]:
### Как подобрать коэффициент регуляризации?

from sklearn.model_selection import GridSearchCV

param_grid = {
    "Lasso__alpha": alphas
}

### Передадим в GridSearchCV

search = GridSearchCV(pipe, param_grid,
                      cv=splitter, scoring='neg_mean_squared_error')

search.fit(X, Y)

print(f"Best parameter (CV score={search.best_score_:.5f}):")
print(search.best_params_)

In [ ]:
### Убедимся, что все ок!

pipe.set_params(Lasso__alpha=search.best_params_['Lasso__alpha'])

In [ ]:
cv_result_pipe = cross_validate(pipe, X, Y,
                                scoring='neg_mean_squared_error',
                                cv=splitter, return_train_score=True)

print(f"Среднее MSLE на тренировочных фолдах: {-np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Среднее MSLE на тестовых фолдах: {-np.mean(cv_result_pipe['test_score']).round(3)}")

### Анализ выбросов

In [ ]:
data = pd.concat((X, Y), axis=1)

In [ ]:
top_quantile = data['log_price_doc'].quantile(0.975)
low_quantile = data['log_price_doc'].quantile(0.025)

print(f"Топ 2,5% значение таргета: {top_quantile.round(2)}")
print(f"Топ 97,5% значение таргета: {low_quantile.round(2)}")

In [ ]:
### Выбросим объекты со значениями вне отрезка [top 2,5%; top97,5%]

data = data[(data['log_price_doc']>low_quantile)&(data['log_price_doc']<top_quantile)]

X_new, Y_new = data.drop('log_price_doc', axis=1), data['log_price_doc']

In [ ]:
### Как подобрать коэффициент регуляризации?

new_splitter = TimeSeriesSplit(n_splits=4)

param_grid = {
    "Lasso__alpha": alphas
}

### Передадим в GridSearchCV

search = GridSearchCV(pipe, param_grid,
                      cv=new_splitter, scoring='neg_mean_squared_error')

search.fit(X_new, Y_new)

print(f"Best parameter (CV score={search.best_score_:.5f}):")
print(search.best_params_)

In [ ]:
### Убедимся, что все ок!

pipe.set_params(Lasso__alpha=search.best_params_['Lasso__alpha'])

In [ ]:
cv_result_pipe = cross_validate(pipe, X_new, Y_new,
                                scoring='neg_mean_squared_error',
                                cv=splitter, return_train_score=True)

print(f"Среднее MSLE на тренировочных фолдах: {-np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Среднее MSLE на тестовых фолдах: {-np.mean(cv_result_pipe['test_score']).round(3)}")

### Новый прием: сегментация данных

In [ ]:
### Разделим квартиры по типу недвижимости
### Для первички и вторички будем строить разные модели

Owner_Occupier = data[data['product_type_OwnerOccupier'] == 1].copy()
Investment = data[data['product_type_OwnerOccupier'] == 0].copy()

In [ ]:
X_Occupier = Owner_Occupier.drop('log_price_doc', axis=1)
X_Investment = Investment.drop('log_price_doc', axis=1)

Y_Occupier = Owner_Occupier['log_price_doc']
Y_Investment = Investment['log_price_doc']

In [ ]:
### Соберем модель для Owner_Occupier

search_Owner_Occupier = GridSearchCV(pipe, param_grid,
                                     cv=splitter, scoring='neg_mean_squared_error')

search_Owner_Occupier.fit(X_Occupier, Y_Occupier)

print(f"Best parameter (CV score={search_Owner_Occupier.best_score_:.5f}):")
print(search_Owner_Occupier.best_params_)

pipe.set_params(Lasso__alpha=search_Owner_Occupier.best_params_['Lasso__alpha'])

cv_result_pipe = cross_validate(pipe, X_Occupier, Y_Occupier,
                                scoring='neg_mean_squared_error',
                                cv=splitter, return_train_score=True)

error_Occupier_train = -np.mean(cv_result_pipe['train_score'])
error_Occupier_test = -np.mean(cv_result_pipe['test_score'])

print(f"Среднее MSLE на тренировочных фолдах: {error_Occupier_train.round(3)}")
print(f"Среднее MSLE на тестовых фолдах: {error_Occupier_test.round(3)}")

In [ ]:
### Соберем модель для Investment

search_Investment = GridSearchCV(pipe, param_grid,
                                cv=splitter, scoring='neg_mean_squared_error')

search_Investment.fit(X_Investment, Y_Investment)

print(f"Best parameter (CV score={search_Investment.best_score_:.5f}):")
print(search_Investment.best_params_)

pipe.set_params(Lasso__alpha=search_Investment.best_params_['Lasso__alpha'])

cv_result_pipe = cross_validate(pipe, X_Investment, Y_Investment,
                                scoring='neg_mean_squared_error',
                                cv=splitter, return_train_score=True)

error_Investment_train = -np.mean(cv_result_pipe['train_score'])
error_Investment_test = -np.mean(cv_result_pipe['test_score'])

print(f"Среднее MSLE на тренировочных фолдах: {error_Investment_train.round(3)}")
print(f"Среднее MSLE на тестовых фолдах: {error_Investment_test.round(3)}")

In [ ]:
### Перевзвесим скоры с учетом количества объектов
### в обоих типах жилья

n_Occupier = Owner_Occupier.shape[0]
n_Investment = Investment.shape[0]

## Посчитаем доли категорий в общий выборке

share_Occupier = n_Occupier / data.shape[0]
share_Investment = n_Investment / data.shape[0]

weighted_error_train = share_Occupier * error_Occupier_train + \
                       share_Investment * error_Investment_train

weighted_error_test = share_Occupier * error_Occupier_test + \
                       share_Investment * error_Investment_test

print(f"Среднее взвешенное MSLE на тренировочных фолдах: {weighted_error_train.round(3)}")
print(f"Среднее взвешенное MSLE на тестовых фолдах: {weighted_error_test.round(3)}")

Как еще можно улучшить модель? Несколько направлений для размышлений:

- Обучить другие модели
- Добавить макропоказатели в датасет
- Сгенерировать новые фичи из уже имеющихся (не простой encoding)
- Првоести более глубокий EDA анализ